In [15]:
import os
import json
import time
import csv
import random
import torch

import numpy as np
import pandas as pd

from openai import OpenAI
from sentence_transformers import SentenceTransformer, util

In [13]:
DOMAINS = [
    "dsa", "pf", "oop", "os", "dbms", "cn",
    "bd", "fd", "sql", "sd", "cicd", "do",
    "ml", "da", "ba", "pm"
]

DIFFICULTY_LEVELS = [0, 1, 2, 3, 4]

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key= user_secrets.get_secret("API_KEY")


In [5]:
client = OpenAI(api_key= api_key,base_url="https://openrouter.ai/api/v1")

MODEL_NAME = "xiaomi/mimo-v2-flash:free"


In [6]:
def call_llm(prompt, max_tokens=600, temperature=0.4):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are an expert computer science interviewer."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    
    return response.choices[0].message.content


In [7]:
QUESTION_PROMPT = """
Generate exactly {num_questions} interview questions with difficulty level {difficulty}.

Each item must be a JSON object with:
1) "question": a single sentence.
   - Must NOT start with any numbering or prefixes.
   - Must NOT contain a question mark (?).
   - Must ALWAYS end with a full stop (.).

2) "difficulty": exactly {difficulty}.

3) "domains": an array chosen ONLY from:
["dsa","pf","oop","os","dbms","cn","bd","fd","sql","sd","cicd","do","ml","da","ba","pm"]

Rules:
- Questions must be technical and interview-relevant.
- Do NOT repeat or rephrase common textbook questions.
- Do NOT drift outside computer science topics.
- Do NOT invent new domain labels.

Return ONLY a JSON array. No explanations.
"""


In [9]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

accepted_questions = []
accepted_embeddings = []

In [10]:
def is_valid_question(item, expected_difficulty):
    if not isinstance(item, dict):
        return False

    if "question" not in item or "domains" not in item or "difficulty" not in item:
        return False

    text = item["question"].strip()

    if "?" in text:
        return False
    if not text.endswith("."):
        return False
    if item["difficulty"] != expected_difficulty:
        return False
    if not item["domains"]:
        return False
    if not all(d in DOMAINS for d in item["domains"]):
        return False
    if len(text.split()) < 6:
        return False

    return True


In [16]:
def is_duplicate(question_text, threshold=0.9):

    if len(accepted_embeddings) == 0:
        return False


    new_emb = embedder.encode(question_text, convert_to_tensor=True)


    existing_embs = torch.stack(accepted_embeddings)

    similarities = util.cos_sim(new_emb, existing_embs)

    return similarities.max().item() > threshold


In [17]:
TARGET_PER_DIFFICULTY = 200
BATCH_SIZE = 10

results = []

for difficulty in DIFFICULTY_LEVELS:
    print(f"\nGenerating questions for difficulty {difficulty}")

    while len([r for r in results if r["difficulty"] == difficulty]) < TARGET_PER_DIFFICULTY:
        
        prompt = QUESTION_PROMPT.format(
            num_questions=BATCH_SIZE,
            difficulty=difficulty
        )

        try:
            raw_output = call_llm(prompt)
            batch = json.loads(raw_output)
        except Exception as e:
            print("Error parsing model output, retrying...")
            time.sleep(2)
            continue

        for item in batch:
            if not is_valid_question(item, difficulty):
                continue

            if is_duplicate(item["question"]):
                continue

            emb = embedder.encode(item["question"], convert_to_tensor=True)

            accepted_questions.append(item["question"])
            accepted_embeddings.append(emb)

            results.append({
                "question": item["question"],
                "domain": ",".join(item["domains"]),
                "difficulty": item["difficulty"]
            })

        print(f"Accepted so far (difficulty {difficulty}):",
              len([r for r in results if r["difficulty"] == difficulty]))

        time.sleep(1)



Generating questions for difficulty 0
Accepted so far (difficulty 0): 10
Accepted so far (difficulty 0): 18
Accepted so far (difficulty 0): 20
Accepted so far (difficulty 0): 28
Accepted so far (difficulty 0): 35
Accepted so far (difficulty 0): 42
Accepted so far (difficulty 0): 47
Accepted so far (difficulty 0): 51
Accepted so far (difficulty 0): 57
Accepted so far (difficulty 0): 62
Accepted so far (difficulty 0): 66
Accepted so far (difficulty 0): 69
Accepted so far (difficulty 0): 76
Accepted so far (difficulty 0): 79
Accepted so far (difficulty 0): 81
Accepted so far (difficulty 0): 81
Accepted so far (difficulty 0): 85
Accepted so far (difficulty 0): 90
Accepted so far (difficulty 0): 94
Accepted so far (difficulty 0): 97
Accepted so far (difficulty 0): 100
Accepted so far (difficulty 0): 102
Accepted so far (difficulty 0): 106
Accepted so far (difficulty 0): 109
Accepted so far (difficulty 0): 116
Accepted so far (difficulty 0): 120
Accepted so far (difficulty 0): 123
Accepted 

In [18]:
df = pd.DataFrame(results)
df.to_csv("xiaomi_mimo_v2.csv", index=False, encoding="utf-8")

print("Saved", len(df), "questions.")

Saved 1014 questions.
